In [1]:
# /// script
# requires-python = ">=3.11,<3.14"
# dependencies = [
#     "azure-identity>=1.19.0",
#     "azure-mgmt-cognitiveservices>=13.5.0",
#     "azure-mgmt-costmanagement>=4.0.0",
#     "azure-ai-projects>=2.0.0",
#     "azure-search-documents>=11.5.0",
#     "python-dotenv>=1.0.0",
# ]
# ///
# Launch with one command (uv builds the environment + kernel for you):
#     uvx juv run m03_ops_tour.ipynb
# Prereqs: `az login`, and a .env in this folder (see .env.example).
print("Ready. Everything below is keyless and safe to re-run.")

Ready. Everything below is keyless and safe to re-run.


# Lesson 3 — Managing and monitoring AI in Foundry (hands-on tour)

Once a Microsoft Foundry solution is live, three questions decide whether it stays healthy:
**do I have capacity, is the model behaving, and is my search data trustworthy?**
This notebook answers all three against a real project, reading **live** signals.

Two things make it safe to lean on:
- **Keyless.** Everything authenticates with your `az login` identity (Microsoft Entra ID) — no API keys anywhere.
- **Idempotent.** Every cell reads or converges to the same state, so you can run the whole thing top to bottom as many times as you like.

You'll also see each of these in the portal; the notebook is the same truth, in code you could automate.

In [2]:
# --- Setup: one credential, read config from .env -------------------------------------
import os
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()  # reads the .env in this folder
cred = AzureCliCredential()  # your `az login` identity

SUBSCRIPTION = os.environ["AZURE_SUBSCRIPTION_ID"]
REGION       = os.environ.get("QUOTA_LOCATION", "eastus2")
PROJECT_EP   = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
SEARCH_EP    = os.environ.get("SEARCH_ENDPOINT", "")
INDEX_NAME   = os.environ.get("SEARCH_INDEX_NAME", "ai103-l03-demo")
print("project :", PROJECT_EP)
print("region  :", REGION, "| search:", SEARCH_EP or "(not set yet)", "| index:", INDEX_NAME)

project : https://aif-ai103-mvp-quc6kxy2j2v2q.services.ai.azure.com/api/projects/proj-lesson-03
region  : eastus2 | search: https://srch-ai103-l03-quc6kxy2j2v2q.search.windows.net | index: ai103-l03-demo


## 1 · Capacity — quota and headroom

Every model deployment draws on a **quota** measured in **tokens per minute (TPM)**, granted
per subscription, per region, per model. Quota is the invisible wall your architecture runs into.

When a deployment creeps toward its limit, that's the cue to **scale** — raise TPM, add PTUs, or
spread traffic across regions — *before* users start seeing throttling (HTTP 429). The number that
matters isn't how much you're using, it's how much **headroom** is left.

A useful way to hold it: on the exam, "a new deployment fails because there's no capacity in the
region" is a **quota** problem, not a code problem. The fix is to free or request TPM in that region.

In [3]:
# Read the model quota for this region -- current usage vs the granted limit (headroom).
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
arm = CognitiveServicesManagementClient(cred, SUBSCRIPTION)

print(f"{'quota':<44}{'used':>8}{'limit':>8}{'headroom':>10}")
print("-" * 70)
for u in arm.usages.list(REGION):
    limit = u.limit or 0
    if limit <= 0:
        continue
    if "gpt-5" not in (u.name.value or "").lower():
        continue  # keep the on-screen list focused on the models this course uses
    used = u.current_value or 0
    name = (u.name.value or "").replace("OpenAI.", "")
    print(f"{name:<44}{used:>8.0f}{limit:>8.0f}{limit-used:>10.0f}")
print("\nRule of thumb: when headroom gets thin, scale or request quota before users feel it.")

quota                                           used   limit  headroom
----------------------------------------------------------------------


GlobalStandard.gpt-5-mini                         20     500       480

Rule of thumb: when headroom gets thin, scale or request quota before users feel it.


### 1b · Cost — the other guardrail

Quota is a **capacity** ceiling; cost is the **spend** ceiling, and you govern it the same way you
watch capacity — by reading it and acting before it bites. The live signal is **month-to-date cost
attributed by workload**: which resource group (and, one level deeper, which deployment) is actually
spending. That's the number you wire to an **Azure budget** with action-group alerts, so you hear
about overspend while there's still time to act.

Two habits ride alongside the read: **tag** every resource so cost slices cleanly by owner, and cap
output length with **`max_tokens`** so one runaway prompt can't generate a novel on your dime.

In [4]:
# Read month-to-date cost, summed and grouped by resource group ("cost per workload").
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryDefinition, QueryDataset, QueryAggregation, QueryGrouping, TimeframeType, ExportType,
)
from azure.core.exceptions import HttpResponseError

query = QueryDefinition(
    type=ExportType.ACTUAL_COST,
    timeframe=TimeframeType.MONTH_TO_DATE,
    dataset=QueryDataset(
        aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
        grouping=[QueryGrouping(type="Dimension", name="ResourceGroup")],
    ),
)
try:
    result = CostManagementClient(cred).query.usage(
        scope=f"/subscriptions/{SUBSCRIPTION}", parameters=query)
    cols = [c.name for c in result.columns]
    i_cost, i_rg, i_cur = cols.index("Cost"), cols.index("ResourceGroup"), cols.index("Currency")
    rows = sorted(result.rows or [], key=lambda r: r[i_cost], reverse=True)
    if rows:
        print(f"Month-to-date cost by resource group ({rows[0][i_cur]}):")
        for r in rows[:8]:
            print(f"  {r[i_rg]:<40}{r[i_cost]:>10.4f}")
        print(f"  {'TOTAL':<40}{sum(r[i_cost] for r in rows):>10.4f}")
    else:
        print("No cost rows this month yet (fresh subscription or no spend).")
except HttpResponseError as exc:
    # Cost Management throttles queries hard (429) and can lag a day on new subs; either is fine here.
    print("Cost read unavailable:", exc.message.splitlines()[0])

print("\nAttach an Azure BUDGET with action-group alerts to get told before the bill lands.")

Month-to-date cost by resource group (USD):
  rg-warnerco-mcp                             0.2055
  rg-ai103-core                               0.1325
  rg-ai103-m01                                0.0083
  flex-9bdf6f5579fe54c27bb5e6d58054a171a9df59b0be9d7aa65f07ae43815e9834    0.0000
  TOTAL                                       0.3463

Attach an Azure BUDGET with action-group alerts to get told before the bill lands.


## 2 · Behavior — the observability pipe

Foundry **observability** is three capabilities working together:
- **Evaluation** — quality and safety scores (coherence, groundedness, content safety).
- **Monitoring** — latency, token use, and error rates on a live dashboard.
- **Tracing** — the step-by-step execution of a call or an agent.

All of it flows through **Azure Monitor Application Insights**. So the very first health check is
simply: *is that pipe connected?* No connection, no signals — which is itself worth knowing, because
a "monitoring shows nothing" problem is almost always a missing or mis-wired Application Insights link.

In [5]:
# Confirm the observability pipe: fetch the project's Application Insights connection.
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ResourceNotFoundError

project = AIProjectClient(endpoint=PROJECT_EP, credential=cred)
try:
    conn = project.telemetry.get_application_insights_connection_string()
    print("Application Insights: CONNECTED")
    print("  ", conn.split(";")[0], "...")   # show only the InstrumentationKey portion
    print("This pipe carries latency, token, safety, and groundedness signals into the dashboard.")
except ResourceNotFoundError:
    print("Application Insights: NOT connected for this project.")
    print("Connect one under the project's Monitoring settings to light up production monitoring.")

Application Insights: CONNECTED
   InstrumentationKey=9eb5e97c-4593-4ee0-ba91-27a67574734e ...
This pipe carries latency, token, safety, and groundedness signals into the dashboard.


## 3 · Data — index health and relevance

A retrieval-augmented app is only as good as the **search index** behind it. When an index quietly
breaks — documents drop out, key fields go empty — the answers get worse with **no error at all**.

So you watch two things:
- **Health** — document count and storage (did ingestion do what you expected?).
- **Relevance** — measured, not assumed. The yardstick is **recall against a labeled gold query set**:
  a handful of queries whose right answers you know in advance. Score how many gold answers land in
  the top *k*, and you can diff a ranking change on hard numbers instead of a hunch.

Below we ensure a small demo index (`ai103-l03-demo`) exists with a few known documents, then read both
signals. Creating the index and adding the documents are both **upserts**, so re-running just converges —
nothing duplicates. (The read-only `m03_ops_demo.py` reads this same index; here we also seed it so the
notebook stands alone.)

In [6]:
# Idempotently ensure the demo index exists with a few KNOWN documents (stable ids = gold answers).
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
)
from azure.search.documents import SearchClient

if not SEARCH_EP:
    print("SEARCH_ENDPOINT not set yet -- skip this cell until the search service is wired into .env.")
else:
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="topic", type=SearchFieldDataType.String, filterable=True, facetable=True),
    ]
    SearchIndexClient(SEARCH_EP, cred).create_or_update_index(  # upsert = idempotent
        SearchIndex(name=INDEX_NAME, fields=fields))

    docs = [
        {"id": "quota-01", "topic": "quota", "title": "Model quota is per-region, per-model TPM",
         "content": "Foundry grants model quota in tokens per minute per subscription, region, and model. "
                    "Raise TPM, add PTUs, or spread regions to scale."},
        {"id": "cost-01", "topic": "cost", "title": "Cost governance with budgets and tags",
         "content": "Tag resources for attribution, set Azure budgets with action-group alerts, track cost "
                    "per deployment, and cap output length with max_tokens."},
        {"id": "ground-01", "topic": "monitoring", "title": "Groundedness scoring on RAG answers",
         "content": "Groundedness measures how well a retrieval-augmented answer sticks to its sources. "
                    "Foundry ships it as a built-in evaluator; a low score signals hallucination."},
        {"id": "relevance-01", "topic": "index", "title": "Relevance with top-k precision and A/B tests",
         "content": "Measure recall against a labeled gold query set, track click-through, monitor freshness, "
                    "dedup stale content, and A/B test ranking changes before rollout."},
    ]
    SearchClient(SEARCH_EP, INDEX_NAME, cred).merge_or_upload_documents(docs)  # upsert = idempotent
    print(f"Index '{INDEX_NAME}' ready with {len(docs)} documents (re-runnable).")

Index 'ai103-l03-demo' ready with 4 documents (re-runnable).


In [7]:
# Read the two signals: HEALTH (doc count, storage) and RELEVANCE (recall over a labeled gold set).
if not SEARCH_EP:
    print("SEARCH_ENDPOINT not set yet -- skip until the search service is wired into .env.")
else:
    stats = SearchIndexClient(SEARCH_EP, cred).get_index_statistics(INDEX_NAME)
    docs_n = stats.get("documentCount", stats.get("document_count", 0))  # key casing varies by API version
    size_n = stats.get("storageSize", stats.get("storage_size", 0))
    print(f"HEALTH    documents: {docs_n}   storage: {size_n} bytes")
    print("          (stats refresh on a delay -- a just-seeded index can read 0 for a minute.)\n")

    # Gold set: each query's right answer is a known id. Recall@k = did that id land in the top k?
    gold = [
        ("how do I scale model quota",                    {"quota-01"}),
        ("budget alerts and cost per deployment",         {"cost-01"}),
        ("groundedness score for RAG answers",            {"ground-01"}),
        ("top-k precision and A/B testing relevance",     {"relevance-01"}),
    ]
    k = 3
    client = SearchClient(SEARCH_EP, INDEX_NAME, cred)
    scores = []
    print(f"RELEVANCE recall@{k} over {len(gold)} labeled queries:")
    for q, expected in gold:
        returned = {h["id"] for h in client.search(search_text=q, top=k, select=["id"])}
        hit = len(expected & returned) / len(expected)
        scores.append(hit)
        print(f"   recall@{k}={hit:.2f}  {q!r}")
    print(f"\nMEAN recall@{k}: {sum(scores)/len(scores):.2f}  (1.00 = every gold answer surfaced)")
    print("A drop here on a ranking change is your signal to A/B test before you ship it.")

HEALTH    documents: 6   storage: 8690 bytes
          (stats refresh on a delay -- a just-seeded index can read 0 for a minute.)

RELEVANCE recall@3 over 4 labeled queries:


   recall@3=1.00  'how do I scale model quota'
   recall@3=1.00  'budget alerts and cost per deployment'
   recall@3=1.00  'groundedness score for RAG answers'
   recall@3=1.00  'top-k precision and A/B testing relevance'

MEAN recall@3: 1.00  (1.00 = every gold answer surfaced)
A drop here on a ranking change is your signal to A/B test before you ship it.


## Recap

The three Lesson 3 questions, answered live against one project:
- **Capacity + cost** — TPM headroom tells you when to scale; month-to-date spend by workload tells you
  what it costs, and a budget alerts you before the bill lands.
- **Behavior** — the Application Insights pipe carries latency, safety, and groundedness.
- **Data** — index health plus recall over a labeled gold set catch a silently degrading RAG pipeline.

Read them on a cadence and you catch trouble before your users do. Everything here was keyless and
idempotent, so it's the same code you'd drop into a scheduled job.